In [ ]:
import pybullet as p
import pybullet_data
import open3d as o3d
import matplotlib.pyplot as plt
import numpy as np
import time
from scipy.spatial import cKDTree
import os

class RobotReachability:
    
    def __init__(self, num_points, urdf_path, mesh_path, mesh_position=[0,0.2,0], mesh_orientation=[0,0,0],base_position=[0,0,0]):
        #Setup PyBullet
        #self.client = p.connect(p.DIRECT)
        self.client = p.connect(p.GUI)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0,0,-9.8)
        
        #Load Robot
        self.robot_id = p.loadURDF(urdf_path, basePosition=base_position, useFixedBase=True)
        self.num_joints = p.getNumJoints(self.robot_id)
        print("-" * 30)
        print(f"Total Joints Found: {self.num_joints}")
        self.ee_index= -1
        self.movable_joints = [] 
        
        for i in range(self.num_joints):
            info = p.getJointInfo(self.robot_id, i)
            joint_name = info[1].decode("utf-8")
            link_name = info[12].decode("utf-8")
            joint_type = info[2]
            
            print(f"ID {i}: Joint='{joint_name}' Link='{link_name}' Type={joint_type}")
            
            # Identify End Effector by name (Usually 'link_6' or 'tool0')
            if "link_6" in link_name or "tool0" in link_name:
                self.ee_index = i
            
            # Store movable joints for IK control
            if joint_type != p.JOINT_FIXED:
                self.movable_joints.append(i)


        if self.ee_index == -1:
            print("WARNING: Could not find 'link_6'. Defaulting to last index.")
            self.ee_index = self.num_joints - 1
            
        print(f"SELECTED END EFFECTOR INDEX: {self.ee_index}")
        print("-" * 30)

        
        #Define Seeds for positional configuration
        self.seeds = {
            "wrist_up" : [0, 0, 0, 0, -1.57, 0],
            "wrist_down" : [0, 0, 0, 0, 1.57, 0]
        }

        #Mesh Processing
        self.mesh=o3d.io.read_triangle_mesh(mesh_path)
        if not self.mesh.has_triangles():
            raise ValueError("CRITICAL ERROR: Failed to load mesh. File path may be wrong.")
        self.mesh.remove_duplicated_vertices()
        self.mesh.remove_duplicated_triangles()
        self.mesh.remove_degenerate_triangles()
        self.mesh.compute_vertex_normals()

        self.mesh.scale(0.001, center=(0,0,0))
        center_offset = self.mesh.get_center()
        self.mesh.translate(-center_offset)
        temp_mesh_path = os.path.abspath("temp_mesh_loader.stl")
        o3d.io.write_triangle_mesh(temp_mesh_path, self.mesh)

        #Load Mesh into PyBullet
        mesh_quat = p.getQuaternionFromEuler(mesh_orientation)
        visual_id = p.createVisualShape(shapeType = p.GEOM_MESH, fileName=temp_mesh_path, rgbaColor=[0.6, 0.6, 0.6, 1], meshScale=[1,1,1])
        collision_id = p.createCollisionShape(shapeType = p.GEOM_MESH, fileName = temp_mesh_path, meshScale=[1,1,1], flags=p.GEOM_FORCE_CONCAVE_TRIMESH)
        
        self.mesh_body_id = p.createMultiBody(
            baseMass=0,
            baseCollisionShapeIndex=collision_id,
            baseVisualShapeIndex=visual_id,
            basePosition=mesh_position,
            baseOrientation=mesh_quat
        )

        rot_matrix_flat=p.getMatrixFromQuaternion(mesh_quat)
        R=np.array(rot_matrix_flat).reshape(3,3)
        self.mesh.rotate(R, center=(0, 0, 0))
        self.mesh.translate(mesh_position)

        self.pcd=self.mesh.sample_points_poisson_disk(number_of_points=num_points)
        self.points = np.asarray(self.pcd.points)
        self.normals= np.asarray(self.pcd.normals)
        print(f"Loaded mesh with with {len(self.points)} subsampled points")

        #Storage for Cell Divisions
        self.signatures=[]
        self.cell_ids=[]
        self.wristup_count=0
        self.wristdown_count=0
        self.max_reach= self.calculate_max_reach()
        self.reachable_mask = self.filter_unreachable_by_distance(self.max_reach)
    
    def generate_all_seeds(self):
        #Not super generalized
        self.seeds={}
        wrist_opts={'wrist_up': -1.57, 'wrist_down': 1.57}
        elbow_opts={'elbow_up': 0}

    def align_vector_to_normal(self,normal):
        #Calculate Quaternion to align robot z-axis to tool
        tool_axis=np.array([0,0,1])
        target_axis = normal
        rotation_axis=np.cross(tool_axis,target_axis)
        if np.linalg.norm(rotation_axis)<1e-6: #already aligned
            if np.dot(tool_axis, target_axis) > 0: 
                return [0, 0, 0, 1]
            return [0,0,0,1] 
                
        rotation_axis= rotation_axis/np.linalg.norm(rotation_axis)
        angle = np.arccos(np.dot(tool_axis, target_axis)) 
        q_xyz=rotation_axis*np.sin(angle/2)
        q_w=np.cos(angle/2)
        return list(q_xyz) + [q_w]   

    def calculate_max_reach(self):
        print("Calculating maximum reach from URDF (summing joint distances)")
        current_joint_index = self.ee_index
        total_length=0.0
        while current_joint_index !=-1:
            joint_info = p.getJointInfo(self.robot_id, current_joint_index)
            parent_link_index=joint_info[16]
            offset_vec= joint_info[14]
            dist=np.linalg.norm(offset_vec)
            #print(f"Link {current_joint_index} is child of Link {parent_link_index}, and is offset by {dist}")
            total_length+=dist
            current_joint_index=parent_link_index  
        #print(f"Calculated Chain Length: {total_length:.3f}m")
        safety_factor = 1.5
        return(total_length*safety_factor)
    
    def filter_unreachable_by_distance(self, max_reach):
        dists=np.linalg.norm(self.points, axis=1)
        mask = dists < (max_reach + 0.1)
        print(f"Distance Culling: {len(self.points)-np.sum(mask)} points ignored due to being completely unreachable")
        return mask
    
    def check_collision(self):
        #Return true if colliding with self or environment
        contact_points=p.getContactPoints(bodyA=self.robot_id, bodyB=self.robot_id)
        for contact in contact_points:
            linkA, linkB = contact[3], contact[4]
            if abs(linkA-linkB) > 1 and contact[8] < -0.005: #Eliminates same-link overlap as false positive
                return True, "Self Collision"
        env_contacts = p.getContactPoints(bodyA=self.robot_id, bodyB=self.mesh_body_id)
        for contact in env_contacts:
            link_index = contact[3] # The link ID on the robot
            distance = contact[8]
            if distance < -0.001: # Genuine collision
                # Get the name of the link for clarity
                if link_index == -1:
                    link_name = "BASE (Fixed Base)"
                else:
                    link_name = p.getJointInfo(self.robot_id, link_index)[12].decode('utf-8')
                
                print(f"CRITICAL COLLISION: {link_name} (ID {link_index}) is inside the part by {abs(distance):.3f}m")
                return True, f"Collision: {link_name}"
            return True, "Part Collision"
        return False, "No Collision"
    
    def check_reachability(self, target_pos, target_normal, seed_name):
        #Set up joint cage
        ll=[]#lower limit
        ul=[]#upper limit
        jr=[]#joint range
        rp=[]#rest pose
        j5_index=4
        seed_conf=self.seeds[seed_name]

        for i, joint_id in enumerate(self.movable_joints):
            info=p.getJointInfo(self.robot_id, joint_id)
            lower=info[8]
            upper=info[9]
            if joint_id==j5_index:
                if seed_name=="wrist_up":
                    upper = -0.05
                elif seed_name=="wrist_down":
                    lower = 0.05
            ll.append(lower)
            ul.append(upper)
            jr.append(upper-lower)
            seed_val = 0
            if i in self.movable_joints:
                m_idx = self.movable_joints.index(i)
                if m_idx < len(seed_conf):
                    seed_val = seed_conf[m_idx]
                rp.append(seed_val)

        for i, joint_id in enumerate(self.movable_joints):
             p.resetJointState(self.robot_id, joint_id, seed_conf[i])

        target_orient = self.align_vector_to_normal(-target_normal)

        # 3. Run IK with the Limits (The Cage)
        joint_poses = p.calculateInverseKinematics(
            self.robot_id,
            self.ee_index,
            target_pos,
            target_orient,
            lowerLimits=ll,    # <--- Enforces J5 > 0 or J5 < 0
            upperLimits=ul,    # <--- Enforces J5 > 0 or J5 < 0
            jointRanges=jr,
            restPoses=rp,
            maxNumIterations=200, 
            residualThreshold=1e-5
        )

        for i, joint_id in enumerate(self.movable_joints):
            p.resetJointState(self.robot_id, joint_id, joint_poses[i])

        actual_pos = p.getLinkState(self.robot_id, self.ee_index)[4]
        dist = np.linalg.norm(np.array(actual_pos) - np.array(target_pos))
        if dist > 0.01: 
            return False, f"Unreachable in {seed_name} ({dist:.3f}m)"
        is_collision, _ = self.check_collision()
        if is_collision:
            return False, "Collision"
        actual_j5 = joint_poses[4] # Index 4 is Joint 5
        if seed_name == "wrist_up" and actual_j5 > 0: return False, "Cage Breach"
        if seed_name == "wrist_down" and actual_j5 < 0: return False, "Cage Breach"
        if seed_name == "wrist_up":
            self.wristup_count+=1
        if seed_name == "wrist_down":
            self.wristdown_count+=1
        return True, "Success"

    def run_analysis(self):
        feasible_mask = self.reachable_mask
        self.signatures=[]
        #print(f"Starting analysis on {len(self.points)} points")
        start_time=time.time()
        for i in range(len(self.points)):
            if not feasible_mask[i]:
                self.signatures.append((False,False))
                continue
            pt = self.points[i]
            nm = self.normals[i]
            up_ok, status =self.check_reachability(pt,nm,"wrist_up")
            down_ok, status =self.check_reachability(pt,nm, "wrist_down")
            self.signatures.append((up_ok,down_ok))

            #if i%200==0: print(f" Processed {i}/{len(self.points)}")
        print(f"Analysis complete in {time.time()-start_time:.2f} seconds")

    def visualize_signatures(self):
        #Colors points strictly by what they can do (Verification Step).
        colors = []
        for sig in self.signatures:
            up, down = sig
            if up and down:
                colors.append([0, 1, 0]) # Green (Both)
            elif up and not down:
                colors.append([1, 1, 0]) # Yellow (Up Only)
            elif down and not up:
                colors.append([0, 0, 1]) # Blue (Down Only)
            else:
                colors.append([1, 0, 0]) # Red (Neither)
        self.pcd.colors = o3d.utility.Vector3dVector(np.array(colors))
        frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
        mesh_wire = o3d.geometry.LineSet.create_from_triangle_mesh(self.mesh)
        o3d.visualization.draw_geometries([self.pcd, mesh_wire, frame])  

    def segment_into_cells(self, radius=0.02):
        #print("Segmeneting surface into cells")
        tree= cKDTree(self.points)
        self.cell_ids = -1 * np.ones(len(self.points), dtype = int)
        current_cell_id=0
        for i in range(len(self.points)):
            if self.cell_ids[i] != -1: continue
            current_sig=self.signatures[i]
            queue=[i]
            self.cell_ids[i] = current_cell_id
            while len(queue)>0:
                idx=queue.pop(0)
                neighbors=tree.query_ball_point(self.points[idx], r=radius)
                for n_idx in neighbors:
                    if self.cell_ids[n_idx] == -1:
                        if self.signatures[n_idx] == current_sig:
                            self.cell_ids[n_idx]=current_cell_id
                            queue.append(n_idx)

            current_cell_id +=1
        print(f"Segmentation complete. Found {current_cell_id+1} unique cells")

    def visualize_result(self):
        num_cells=self.cell_ids.max() +1
        cell_colors_map = np.random.uniform(0,1, (num_cells,3))
        final_colors=[]
        for i in range((len(self.points))):
            if self.signatures[i] == (False,False):
                final_colors.append([1,0,0])
            else:
                c_id=self.cell_ids[i]
                final_colors.append(cell_colors_map[c_id])
        self.pcd.colors=o3d.utility.Vector3dVector(np.array(final_colors))
        mesh_wire=o3d.geometry.LineSet.create_from_triangle_mesh(self.mesh)
        o3d.visualization.draw_geometries([self.pcd, mesh_wire])

    def verify_poses_visually(self):
            """
            Re-plays specific successful points to visually confirm the 
            Wrist Up vs Wrist Down logic.
            """
            print("\n" + "="*40)
            print("VISUAL VERIFICATION MODE")
            print("="*40)
            
            # 1. Sort indices by category
            up_only_indices = []
            down_only_indices = []
            both_indices = []
            
            for i, sig in enumerate(self.signatures):
                if sig == (True, False): up_only_indices.append(i)
                elif sig == (False, True): down_only_indices.append(i)
                elif sig == (True, True): both_indices.append(i)

            print(f"Found: {len(up_only_indices)} Up-Only, {len(down_only_indices)} Down-Only, {len(both_indices)} Both.")
            
            # 2. Define a helper to replay a specific point
            def show_pose(idx, mode_name, seed_key):
                pt = self.points[idx]
                nm = self.normals[idx]
                print(f"  Showing Point {idx} -> {mode_name}...")
                
                line_id = p.addUserDebugLine(pt, pt + nm*0.1, [1, 0, 1], lineWidth=3, lifeTime=2.0)
                self.check_reachability(pt, nm, seed_key)
                time.sleep(1.0)

            # 3. Visualize "Down Only" 
            if len(down_only_indices) > 0:
                print("\n--- REPLAYING 'WRIST DOWN' ONLY ---")
                # Show 5 examples
                for i in down_only_indices[:5]:
                    show_pose(i, "Wrist DOWN", "wrist_down")

            # 4. Visualize "Up Only"
            if len(up_only_indices) > 0:
                print("\n--- REPLAYING 'WRIST UP' ONLY ---")
                for i in up_only_indices[:5]:
                    show_pose(i, "Wrist UP", "wrist_up")
                    
            # 5. Visualize "Both" (Show the flip!)
            if len(both_indices) > 0:
                print("\n--- REPLAYING 'BOTH' (Watch the Flip!) ---")
                for i in both_indices[:3]:
                    print(f"  Point {i}: Flipping...")
                    show_pose(i, "Wrist UP", "wrist_up")
                    time.sleep(0.5)
                    show_pose(i, "Wrist DOWN", "wrist_down")
                    time.sleep(1.0)

            print("\nVerification Complete.")

if __name__ == "__main__":
    abb_urdf_file = r"C:\Users\jonas\BARC_NDI\pointcloudcpp\abbIrb120.urdf" 
    ur_urdf_file=r"C:\Users\jonas\BARC_NDI\pointcloudcpp\ur5.urdf"
    mesh_file = r"C:\Users\jonas\BARC_NDI\pointcloudcpp\plane_segments\Airfoil_Surface_example.stl"
    
    # Placement: 0.5m forward, 0.2m up. Rotated upright.
    part_pos = [0.5, 0, 0.3] 
    part_euler = [0, 1.57, 3.14] 
    num_points=8000
    # --- EXECUTION ---
    segmenter = RobotReachability(num_points, abb_urdf_file, mesh_file, mesh_position=part_pos, mesh_orientation=part_euler)
    
    # 1. Reachability Check
    segmenter.run_analysis()
    
    #segmenter.verify_poses_visually()
    segmenter.visualize_signatures()
    # 2. Cell Formation (Radius = 2cm neighbor search)
    segmenter.segment_into_cells(radius=0.05)
    
    # 3. View Results
    segmenter.visualize_result()
    print(f"Wrist up successes: {segmenter.wristup_count}")
    print(f"Wrist down successes: {segmenter.wristdown_count}")
    if p.isConnected():
        p.disconnect()



------------------------------
Total Joints Found: 8
ID 0: Joint='joint_1' Link='link_1' Type=0
ID 1: Joint='joint_2' Link='link_2' Type=0
ID 2: Joint='joint_3' Link='link_3' Type=0
ID 3: Joint='joint_4' Link='link_4' Type=0
ID 4: Joint='joint_5' Link='link_5' Type=0
ID 5: Joint='joint_6' Link='link_6' Type=0
ID 6: Joint='joint6-tool0' Link='tool0' Type=4
ID 7: Joint='base_link-base' Link='base' Type=4
SELECTED END EFFECTOR INDEX: 6
------------------------------
Loaded mesh with with 10000 subsampled points
Calculating maximum reach from URDF (summing joint distances)
Distance Culling: 5055 points ignored due to being completely unreachable
Analysis complete in 18.79 seconds
Segmentation complete. Found 8 unique cells
Wrist up successes: 710
Wrist down successes: 155
